# Week 2 — Featurization + Baseline Models

Turns each molecule's SMILES string into a numeric feature vector a model can consume, then trains a baseline random forest per property.

Two representations, combined:
- **Morgan fingerprints** — circular substructure fingerprints, the standard cheminformatics baseline. Captures fine-grained structural detail but isn't human-interpretable.
- **Lipinski-style descriptors** (MolWt, LogP, HBD, HBA, rotatable bonds) — coarser, but each one has a direct chemical meaning, which is useful later for feature-importance interpretation (Week 3).

Reuses `largest_fragment()` from `Data_EDA.ipynb` so salt-stripping stays consistent with the EDA pass.

**Sealed as of Week 2's review (2026-08-27):** kept frozen and unedited from here on, so it stays a stable "before" snapshot to compare Week 3's improvements against in `Model_Improvement.ipynb` (which copies these functions rather than importing them, for exactly that reason).

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.DataStructs import ConvertToNumpyArray


# Strips counter-ions/solvents from a salt, keeping only the largest fragment
# (by heavy-atom count) — e.g. Clofilium phosphate -> just the clofilium cation.
# Reused from Data_EDA.ipynb so salt-stripping stays consistent with the EDA pass.
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) == 1:
        return mol
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())


# Encodes a molecule's local substructures as a fixed-length bit vector, so a
# molecule of any size/shape becomes a feature array of constant length (n_bits).
def compute_morgan_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

In [2]:
# Converts a whole dataframe of SMILES into one feature matrix ready for
# scikit-learn: fingerprint bits + interpretable descriptors, side by side.
def featurize(df, smiles_col='Drug', radius=2, n_bits=2048):
    fp_rows = []
    desc_rows = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        mol = largest_fragment(mol)
        fp_rows.append(compute_morgan_fp(mol, radius=radius, n_bits=n_bits))
        desc_rows.append({
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        })

    fp_cols = [f'fp_{i}' for i in range(n_bits)]
    fp_df = pd.DataFrame(np.vstack(fp_rows), columns=fp_cols, index=df.index)
    desc_df = pd.DataFrame(desc_rows, index=df.index)
    return pd.concat([fp_df, desc_df], axis=1)

In [3]:
# Deduplicates a split on SMILES: if the same molecule appears more than once
# with *conflicting* labels, we can't trust either copy, so both are dropped;
# if it appears more than once with the *same* label, only one copy is kept
# so the model (and the reported metrics) don't double-count that molecule.
# This is the Week 1 "duplicate SMILES / label conflict" cleanup, applied now
# at modeling time as planned.
def dedupe_labels(df, smiles_col='Drug', y_col='Y'):
    label_counts = df.groupby(smiles_col)[y_col].nunique()
    conflicting = label_counts[label_counts > 1].index
    clean = df[~df[smiles_col].isin(conflicting)]
    return clean.drop_duplicates(subset=smiles_col)

In [4]:
# Regression counterpart to dedupe_labels: for a continuous target, two
# repeat measurements of the same molecule almost never come out to the
# exact same float even when the true value is the same, so treating any
# difference as a "conflict" and dropping would discard real data (and did,
# on Clearance: 16% of train). Instead, average Y across duplicate SMILES,
# treating repeats as noisy measurements of one true value.
def average_duplicate_targets(df, smiles_col='Drug', y_col='Y'):
    other_cols = [c for c in df.columns if c not in (smiles_col, y_col)]
    agg = {c: 'first' for c in other_cols}
    agg[y_col] = 'mean'
    return df.groupby(smiles_col, as_index=False).agg(agg)

In [5]:
# Loads a TDC dataset's scaffold split and applies a dedup/cleanup function
# to both train and test. dedup_fn defaults to dedupe_labels (classification:
# drop genuine conflicts); pass average_duplicate_targets for regression
# targets instead, where "different value" usually means noise, not conflict.
def load_scaffold_split(dataset_class, name, dedup_fn=dedupe_labels):
    data = dataset_class(name=name)
    split = data.get_split(method='scaffold')
    train_df = dedup_fn(split['train'])
    test_df = dedup_fn(split['test'])
    return train_df, test_df

In [6]:
import numpy as np
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, balanced_accuracy_score,
)


# Regression evaluation (Solubility, Clearance): RMSE and MAE are in the
# same units as Y, R^2 is the fraction of variance explained. MAE matches
# what the TDC leaderboard reports, for direct comparison.
def evaluate_regression(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"R^2:  {r2:.3f}")
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


# Classification evaluation (BBB, hERG, CYP3A4): ROC-AUC uses predicted
# probabilities (ranking quality across every threshold); F1 and balanced
# accuracy use the hard 0/1 prediction. Chosen over raw accuracy because
# these datasets are class-imbalanced (see NOTES.md).
def evaluate_classification(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    print(f"ROC-AUC:           {roc_auc:.3f}")
    print(f"F1:                {f1:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    return {'roc_auc': roc_auc, 'f1': f1, 'balanced_accuracy': bal_acc}

## Solubility — baseline model

Proving the pipeline end-to-end on the cleanest dataset first (no missing data, no label conflicts). Load via TDC's **scaffold split**, featurize, train a baseline random forest, evaluate.

In [7]:
from tdc.single_pred import ADME

# Reusable loader: scaffold split + dedupe_labels cleanup, applied the same
# way for every property.
train_df, test_df = load_scaffold_split(ADME, 'Solubility_AqSolDB')
print(f"train: {train_df.shape[0]} molecules, test: {test_df.shape[0]} molecules")

Found local copy...


Loading...


Done!



  0%|          | 0/9982 [00:00<?, ?it/s]


 12%|█▏        | 1233/9982 [00:00<00:00, 11793.06it/s]

...


 66%|██████▌   | 6612/9982 [00:00<00:00, 12883.29it/s]


 81%|████████  | 8096/9982 [00:00<00:00, 13514.56it/s]


 95%|█████████▍| 9477/9982 [00:00<00:00, 13149.80it/s]


100%|██████████| 9982/9982 [00:00<00:00, 12248.65it/s]

train: 6987 molecules, test: 1997 molecules


In [8]:
# Featurize both splits with the function defined above (fingerprints + descriptors).
# This is the slow step — computing a 2048-bit fingerprint per molecule.
X_train = featurize(train_df)
X_test = featurize(test_df)

y_train = train_df['Y'].values
y_test = test_df['Y'].values

X_train.shape, X_test.shape

((6987, 2053), (1997, 2053))

In [9]:
from sklearn.ensemble import RandomForestRegressor

# Baseline model: default hyperparameters aside from n_estimators, so we get an
# honest "no tuning" reference point before Week 3's tuning/comparison pass.
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
# Evaluate on the held-out scaffold-split test set — molecules structurally
# unseen during training.
_ = evaluate_regression(rf, X_test, y_test)

RMSE: 1.286
MAE:  0.926
R^2:  0.686


## BBB Permeability — baseline model

Reusing the same `featurize()` pipeline, but this is a **classification** task (permeable vs. not), so the model and metrics change: `RandomForestClassifier` instead of `Regressor`, and ROC-AUC instead of RMSE/R² — chosen back in Week 1 EDA because the classes are imbalanced (77% permeable / 23% not), which makes raw accuracy misleading.

In [11]:
from tdc.single_pred import ADME

# Same reusable loader as Solubility — scaffold split + dedupe_labels cleanup
# built in, so BBB's Week 1 duplicate/label-conflict findings get handled
# automatically rather than as a separate hand-written step.
bbb_train_df, bbb_test_df = load_scaffold_split(ADME, 'BBB_Martins')
print(f"train: {bbb_train_df.shape[0]} molecules, test: {bbb_test_df.shape[0]} molecules")
print(bbb_train_df['Y'].value_counts(normalize=True))

Found local copy...


Loading...


Done!



  0%|          | 0/2030 [00:00<?, ?it/s]


 40%|███▉      | 806/2030 [00:00<00:00, 8047.11it/s]


 79%|███████▉  | 1611/2030 [00:00<00:00, 7797.23it/s]


100%|██████████| 2030/2030 [00:00<00:00, 8023.71it/s]

train: 1376 molecules, test: 392 molecules
Y
1    0.742006
0    0.257994
Name: proportion, dtype: float64


In [12]:
# Featurize both splits — same function as Solubility, no changes needed since
# it works off SMILES + salt-stripping regardless of the target type.
X_bbb_train = featurize(bbb_train_df)
X_bbb_test = featurize(bbb_test_df)

y_bbb_train = bbb_train_df['Y'].values
y_bbb_test = bbb_test_df['Y'].values

X_bbb_train.shape, X_bbb_test.shape

((1376, 2053), (392, 2053))

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Classifier instead of regressor. class_weight='balanced' re-weights the
# minority class (non-permeable, 23%) during training so the model doesn't
# just learn to predict "permeable" for everything and still look accurate.
rf_bbb = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_bbb.fit(X_bbb_train, y_bbb_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
# Evaluate on the held-out scaffold-split test set.
_ = evaluate_classification(rf_bbb, X_bbb_test, y_bbb_test)

ROC-AUC:           0.917
F1:                0.934
Balanced accuracy: 0.767


## hERG Blockade — baseline model

Classification task, same as BBB, but loaded via TDC's `Tox` class rather than `ADME` (hERG is a toxicity endpoint). Class balance is skewed the other way from BBB — 68% blockers (`Y=1`) vs. 32% non-blockers (`Y=0`) — and Week 1 EDA found 5 duplicate SMILES groups in train (2 with conflicting labels, both sertindole-related analogs) plus a small amount of train/test leakage on the random split. `load_scaffold_split()` handles the scaffold split + label-conflict cleanup the same way it did for BBB.

In [15]:
from tdc.single_pred import Tox

herg_train_df, herg_test_df = load_scaffold_split(Tox, 'hERG')
print(f"train: {herg_train_df.shape[0]} molecules, test: {herg_test_df.shape[0]} molecules")
print(herg_train_df['Y'].value_counts(normalize=True))

Found local copy...


Loading...


Done!



  0%|          | 0/655 [00:00<?, ?it/s]


100%|██████████| 655/655 [00:00<00:00, 7185.48it/s]

train: 456 molecules, test: 125 molecules
Y
1.0    0.684211
0.0    0.315789
Name: proportion, dtype: float64


In [16]:
X_herg_train = featurize(herg_train_df)
X_herg_test = featurize(herg_test_df)

y_herg_train = herg_train_df['Y'].values
y_herg_test = herg_test_df['Y'].values

X_herg_train.shape, X_herg_test.shape

((456, 2053), (125, 2053))

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_herg = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_herg.fit(X_herg_train, y_herg_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
# Evaluate on the held-out scaffold-split test set.
_ = evaluate_classification(rf_herg, X_herg_test, y_herg_test)

ROC-AUC:           0.851
F1:                0.910
Balanced accuracy: 0.739


## CYP3A4 Inhibition — baseline model

Classification task, loaded via `ADME(name='CYP3A4_Veith')`. This is the largest and cleanest dataset in the panel — Week 1 EDA found zero duplicate SMILES and zero train/valid/test leakage, so `load_scaffold_split()`'s dedup step should be a no-op here, same as it was for Solubility. Class balance is close to even (59% non-inhibitor / 41% inhibitor), the most balanced of the three classification tasks so far.

In [19]:
from tdc.single_pred import ADME

cyp3a4_train_df, cyp3a4_test_df = load_scaffold_split(ADME, 'CYP3A4_Veith')
print(f"train: {cyp3a4_train_df.shape[0]} molecules, test: {cyp3a4_test_df.shape[0]} molecules")
print(cyp3a4_train_df['Y'].value_counts(normalize=True))

Found local copy...


Loading...


Done!



  0%|          | 0/12328 [00:00<?, ?it/s]


  6%|▌         | 758/12328 [00:00<00:01, 7573.42it/s]

...


 86%|████████▌ | 10551/12328 [00:01<00:00, 7635.12it/s]


 92%|█████████▏| 11319/12328 [00:01<00:00, 7645.17it/s]


 98%|█████████▊| 12087/12328 [00:01<00:00, 7359.08it/s]


100%|██████████| 12328/12328 [00:01<00:00, 7711.72it/s]

train: 8629 molecules, test: 2467 molecules
Y
0    0.596941
1    0.403059
Name: proportion, dtype: float64


In [20]:
X_cyp3a4_train = featurize(cyp3a4_train_df)
X_cyp3a4_test = featurize(cyp3a4_test_df)

y_cyp3a4_train = cyp3a4_train_df['Y'].values
y_cyp3a4_test = cyp3a4_test_df['Y'].values

X_cyp3a4_train.shape, X_cyp3a4_test.shape

((8629, 2053), (2467, 2053))

In [21]:
from sklearn.ensemble import RandomForestClassifier

# Class balance here is close to even (59/41), so class_weight='balanced'
# matters less than it did for BBB/hERG, but kept for consistency.
rf_cyp3a4 = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_cyp3a4.fit(X_cyp3a4_train, y_cyp3a4_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
from sklearn.metrics import average_precision_score

# Evaluate on the held-out scaffold-split test set. AUPRC (area under the
# precision-recall curve) is added here specifically because the CYP3A4_Veith
# TDC leaderboard reports AUPRC, not ROC-AUC — needed for a fair comparison.
_ = evaluate_classification(rf_cyp3a4, X_cyp3a4_test, y_cyp3a4_test)

y_cyp3a4_proba = rf_cyp3a4.predict_proba(X_cyp3a4_test)[:, 1]
auprc = average_precision_score(y_cyp3a4_test, y_cyp3a4_proba)
print(f"AUPRC:             {auprc:.3f}")

ROC-AUC:           0.879
F1:                0.750
Balanced accuracy: 0.779
AUPRC:             0.852


## Clearance — baseline model

Regression task, the messiest dataset in the panel per Week 1 EDA: heavily right-skewed target, assay censoring (16% of rows pinned at exactly `Y=3.0`, 11% at `Y=150.0` — almost certainly `<3`/`>150` assay boundary values, not real point measurements), and by far the heaviest duplication (94/849 duplicate SMILES on the random split). Uses `average_duplicate_targets` instead of `dedupe_labels` for the reason documented above — checked directly on this scaffold split, every duplicate-SMILES group here had a *different* `Y` value between copies, consistent with real measurement noise rather than genuine conflicts.

The assay censoring itself isn't fixed here (per Week 1's decision to defer, not silently patch) — a plain regressor will be scored as if the pinned values were exact measurements, which is a real limitation worth keeping in mind when reading the result below.

In [23]:
from tdc.single_pred import ADME

clearance_train_df, clearance_test_df = load_scaffold_split(
    ADME, 'Clearance_Hepatocyte_AZ', dedup_fn=average_duplicate_targets
)
print(f"train: {clearance_train_df.shape[0]} molecules, test: {clearance_test_df.shape[0]} molecules")

Found local copy...


Loading...


Done!



  0%|          | 0/1213 [00:00<?, ?it/s]


 53%|█████▎    | 642/1213 [00:00<00:00, 6414.93it/s]


100%|██████████| 1213/1213 [00:00<00:00, 6544.48it/s]

train: 713 molecules, test: 203 molecules


In [24]:
X_clearance_train = featurize(clearance_train_df)
X_clearance_test = featurize(clearance_test_df)

y_clearance_train = clearance_train_df['Y'].values
y_clearance_test = clearance_test_df['Y'].values

X_clearance_train.shape, X_clearance_test.shape

((713, 2053), (203, 2053))

In [25]:
from sklearn.ensemble import RandomForestRegressor

rf_clearance = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_clearance.fit(X_clearance_train, y_clearance_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
from scipy.stats import spearmanr

# Evaluate on the held-out scaffold-split test set. Spearman correlation
# added specifically because the Clearance_Hepatocyte_AZ TDC leaderboard
# reports Spearman, not R^2 — needed for a fair comparison. Spearman measures
# rank agreement (does the model order molecules correctly, high-to-low
# clearance) rather than exact value match, which matters a lot here given
# the assay-censoring pattern distorting exact values at the extremes.
_ = evaluate_regression(rf_clearance, X_clearance_test, y_clearance_test)

y_clearance_pred = rf_clearance.predict(X_clearance_test)
spearman_corr, _ = spearmanr(y_clearance_test, y_clearance_pred)
print(f"Spearman:          {spearman_corr:.3f}")

RMSE: 44.480
MAE:  35.069
R^2:  0.115
Spearman:          0.362


### Comparison: `average_duplicate_targets` vs. `dedupe_labels` on Clearance

Before settling on `average_duplicate_targets` for Clearance's duplicate SMILES, it's worth actually testing the alternative rather than just arguing for it — `dedupe_labels()` (the drop-conflicts approach used for BBB/hERG/CYP3A4's binary targets) *could* technically be applied here too, since it doesn't care whether the target is binary or continuous, it just checks whether `Y` differs across duplicate rows. The question is whether that's the *right* choice for a continuous target, not just whether it runs.

Re-running the identical pipeline (same `featurize()`, same `RandomForestRegressor` config) with `dedupe_labels` swapped in for `average_duplicate_targets`, changing nothing else:

In [ ]:
dropconflicts_train_df, dropconflicts_test_df = load_scaffold_split(
    ADME, 'Clearance_Hepatocyte_AZ', dedup_fn=dedupe_labels
)
print(f"train: {dropconflicts_train_df.shape[0]} molecules, test: {dropconflicts_test_df.shape[0]} molecules  (dedupe_labels)")

X_dropconflicts_train = featurize(dropconflicts_train_df)
X_dropconflicts_test = featurize(dropconflicts_test_df)
y_dropconflicts_train = dropconflicts_train_df['Y'].values
y_dropconflicts_test = dropconflicts_test_df['Y'].values

rf_dropconflicts = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_dropconflicts.fit(X_dropconflicts_train, y_dropconflicts_train)

_ = evaluate_regression(rf_dropconflicts, X_dropconflicts_test, y_dropconflicts_test)
dropconflicts_spearman, _ = spearmanr(y_dropconflicts_test, rf_dropconflicts.predict(X_dropconflicts_test))
print(f"Spearman:          {dropconflicts_spearman:.3f}")

Found local copy...


Loading...


Done!



  0%|          | 0/1213 [00:00<?, ?it/s]


 54%|█████▍    | 661/1213 [00:00<00:00, 6604.01it/s]


100%|██████████| 1213/1213 [00:00<00:00, 6588.96it/s]

train: 577 molecules, test: 163 molecules  (dedupe_labels)


RMSE: 46.915
MAE:  36.939
R^2:  0.125
Spearman:          0.334


**Result:** `dedupe_labels` shrinks train from 849 raw rows down to 577 molecules (713 with averaging) — it doesn't just remove duplicate *rows*, it discards ~136 distinct *molecules* entirely, since every duplicate-SMILES group in this dataset had a differing `Y` (see the reasoning above and in `NOTES.md`). On RMSE/MAE, dropping looks slightly worse; R² comes out marginally *higher* for dropping, but on **Spearman — the actual metric this dataset's TDC leaderboard uses** — averaging wins (0.362 vs. 0.334). Since Spearman is what the comparison against the leaderboard actually depends on, and averaging also keeps ~16% more distinct molecules in training, `average_duplicate_targets` is kept as the pipeline's approach for Clearance; this comparison stays here as evidence for that choice, not just an argument for it.